# The Assistant Axis — A Beginner's Guide to the Entire Codebase

**What this notebook covers:** This is a complete walkthrough of the `assistant-axis` project — what it does, why it exists, how all the pieces fit together, and how to actually run it. If the code feels scattered, this notebook is the map.

---

## Table of Contents

1. [The Big Idea](#1-the-big-idea)
2. [Key Concepts You Need First](#2-key-concepts-you-need-first)
3. [Architecture Overview — How the Files Connect](#3-architecture-overview)
4. [The 5-Step Pipeline — End to End](#4-the-5-step-pipeline)
5. [Core Module Deep Dives](#5-core-module-deep-dives)
6. [Internals — The Engine Room](#6-internals)
7. [Using the Axis — Projection & Steering](#7-using-the-axis)
8. [PCA Analysis — Is the Axis Real?](#8-pca-analysis)
9. [Complete Control Flow — Trace a Request Start to Finish](#9-complete-control-flow)
10. [Quick Reference — What Each File Does](#10-quick-reference)

---

## 1. The Big Idea <a id='1-the-big-idea'></a>

### The Question This Project Answers

When you tell a language model like Gemma or LLaMA *"You are a pirate captain"*, it starts talking like a pirate. When you don't give it any special instructions, it behaves like a helpful AI assistant. **What actually changes inside the model?**

This project discovers that the change lives in a specific **direction** in the model's internal representation space. That direction is called the **Assistant Axis**.

### The Core Formula

```
axis = mean(default_activations) - mean(role_playing_activations)
```

That's it. The axis is literally: *"What does the model look like when it's being a normal assistant?"* minus *"What does the model look like when it's fully role-playing?"*

### What Can You Do With It?

1. **Measure** where a model sits on the role-playing ↔ assistant spectrum (projection)
2. **Steer** a model to be more or less assistant-like in real time (activation steering)
3. **Cap** how far a model drifts from its assistant behavior (activation capping)
4. **Analyze** the structure of role-playing in activation space (PCA)

### Analogy

Think of it like a compass needle. The axis is the compass — it points from "fully role-playing" toward "fully assistant." Given any model response, you can check where the needle points to see how much the model has drifted from its default behavior.

---

## 2. Key Concepts You Need First <a id='2-key-concepts-you-need-first'></a>

### Activations (Hidden States)

A transformer model processes text through a stack of **layers** (e.g., Gemma-2-27B has 46 layers). At each layer, every token has a **hidden state** — a vector of numbers (e.g., 3584 dimensions for Gemma). These vectors encode what the model "thinks" about the text at that point.

```
Input: "Hello, how can I help you?"
         ↓ passes through 46 layers
At layer 22, token "help" → [0.23, -1.4, 0.07, ...] (3584 numbers)
```

**Shape convention:** Throughout this codebase, activations typically have shape:
- Single sample: `(n_layers, hidden_dim)` — one vector per layer
- Batch: `(n_samples, n_layers, hidden_dim)`
- Full sequence: `(n_layers, n_tokens, hidden_dim)` — one vector per token per layer

### Roles vs. Default Behavior

- **Role-playing**: The model is given a system prompt like *"You are a pirate captain"* and fully adopts that persona
- **Default/Assistant**: The model has no special persona — it's just being a helpful AI assistant

The project uses **277 different roles** (pirate, doctor, philosopher, etc.) and measures how each one changes the model's activations.

### Scoring (0-3 Scale)

Not every response to a role prompt actually plays the role well. An LLM judge (GPT-4) scores each response:

| Score | Meaning |
|-------|--------|
| 0 | Model refused to answer |
| 1 | Model answered but stayed as "AI assistant" |
| 2 | Model mixed — identifies as AI but shows some role traits |
| 3 | Model is **fully** playing the role |

Only score=3 responses are used to compute the axis, because those are the ones where the model truly adopted the persona.

### Projection and Steering

- **Projection**: Take a dot product of an activation with the axis to get a single number. High = more assistant-like. Low = more role-playing.
- **Steering**: During generation, add or subtract the axis direction from the model's activations to push behavior one way or the other.

---

## 3. Architecture Overview — How the Files Connect <a id='3-architecture-overview'></a>

Here's the directory structure with purpose annotations:

```
assistant-axis/
│
├── assistant_axis/                    # ← THE LIBRARY (importable Python package)
│   ├── __init__.py                    #    Public API — what you import
│   ├── axis.py                        #    Compute, save, load, project onto the axis
│   ├── generation.py                  #    Generate model responses (vLLM batch + HuggingFace)
│   ├── judge.py                       #    Score responses with an LLM judge (GPT-4)
│   ├── models.py                      #    Model configs (which layer to use, etc.)
│   ├── pca.py                         #    PCA analysis and visualization
│   ├── steering.py                    #    Modify model behavior at inference time
│   │
│   ├── internals/                     #    Low-level model interaction
│   │   ├── model.py                   #      ProbingModel — wraps HuggingFace models
│   │   ├── activations.py             #      Extract hidden states from model layers
│   │   ├── conversation.py            #      Chat template formatting + tokenization
│   │   ├── spans.py                   #      Map token positions to activations
│   │   └── exceptions.py              #      Custom exceptions
│   │
│   └── tests/                         #    Unit tests
│       ├── test_axis.py               #      Tests for axis computation
│       └── test_generation.py         #      Tests for generation utilities
│
├── pipeline/                           # ← THE PIPELINE (scripts to run end-to-end)
│   ├── 1_generate.py                  #    Step 1: Generate responses for all roles
│   ├── 2_activations.py               #    Step 2: Extract activations from responses
│   ├── 3_judge.py                     #    Step 3: Score responses with GPT-4
│   ├── 4_vectors.py                   #    Step 4: Compute per-role mean vectors
│   ├── 5_axis.py                      #    Step 5: Compute the final axis
│   └── run_pipeline.sh                #    Shell script to run all 5 steps
│
├── data/                              # ← INPUT DATA
│   ├── roles/instructions/*.json      #    277 role definitions (system prompts)
│   ├── traits/instructions/*.json     #    240 trait definitions
│   └── extraction_questions.jsonl     #    Questions asked to each role
│
├── notebooks/                         # ← ANALYSIS & DEMOS
│   ├── pca.ipynb                      #    PCA analysis of activation space
│   ├── steer.ipynb                    #    Steering demo
│   ├── project_transcipt.ipynb        #    Project a conversation onto the axis
│   └── visualize_axis.ipynb           #    Visualize axis properties
│
└── transcripts/                       # ← EXAMPLE OUTPUTS
    ├── case_studies/                   #    Before/after steering examples
    └── persona_drift/                 #    Conversations showing persona drift
```

### The Two Halves

The codebase has two distinct halves:

1. **The Pipeline** (`pipeline/`): Run once to create the axis. It's a data-processing pipeline — generate data, extract features, score, aggregate, compute.

2. **The Library** (`assistant_axis/`): Use the axis. Import it in notebooks or scripts to measure, steer, or analyze models.

The pipeline *produces* the axis. The library *uses* it.

---

## 4. The 5-Step Pipeline — End to End <a id='4-the-5-step-pipeline'></a>

This is the heart of the project. Here's the full flow:

```
  277 Role JSONs + 240 Questions
           │
           ▼
  ┌──────────────────┐
  │  1. GENERATE      │  For each role × each question, generate a model response
  │  (1_generate.py)  │  using vLLM batch inference. ~277 × 240 = 66,480 responses.
  └──────────────────┘
           │
           ▼  responses/*.jsonl
  ┌──────────────────┐
  │  2. ACTIVATIONS   │  Run each response through the model again with hooks to
  │  (2_activations)  │  capture the hidden state at every layer.
  └──────────────────┘
           │
           ▼  activations/*.pt  (one per role)
  ┌──────────────────┐
  │  3. JUDGE         │  Use GPT-4 to score each response on the 0-3 scale.
  │  (3_judge.py)     │  "Did the model actually play the role?"
  └──────────────────┘
           │
           ▼  scores/*.json  (one per role)
  ┌──────────────────┐
  │  4. VECTORS       │  For each role, take the mean of score=3 activations.
  │  (4_vectors.py)   │  For default: take the mean of ALL activations.
  └──────────────────┘
           │
           ▼  vectors/*.pt  (one per role)
  ┌──────────────────┐
  │  5. AXIS          │  axis = mean(default_vectors) - mean(role_vectors)
  │  (5_axis.py)      │  That's it — one subtraction.
  └──────────────────┘
           │
           ▼  axis.pt  ← THIS IS THE FINAL OUTPUT
```

### Why So Many Steps?

The axis formula is simple (`mean_default - mean_role`), but getting good data to compute it is hard:

- **Step 1** is needed because you need diverse responses from many roles and questions
- **Step 2** is needed because activations aren't stored during normal inference
- **Step 3** is needed because not all responses actually play the role (some refuse, some half-commit)
- **Step 4** averages out noise within each role
- **Step 5** averages across roles to get a universal direction

### How to Actually Run It

```bash
# The simplest way — edit variables in the script first
./pipeline/run_pipeline.sh

# Or step by step:
uv run pipeline/1_generate.py --model "Qwen/Qwen3-32B" --output_dir outputs/responses
uv run pipeline/2_activations.py --model "Qwen/Qwen3-32B" --responses_dir outputs/responses --output_dir outputs/activations
uv run pipeline/3_judge.py --responses_dir outputs/responses --output_dir outputs/scores
uv run pipeline/4_vectors.py --activations_dir outputs/activations --scores_dir outputs/scores --output_dir outputs/vectors
uv run pipeline/5_axis.py --vectors_dir outputs/vectors --output outputs/axis.pt
```

**Important:** Steps 1 and 2 require GPU(s) and take hours. Step 3 requires an OpenAI API key. Steps 4 and 5 are fast (CPU-only, seconds).

---

## 5. Core Module Deep Dives <a id='5-core-module-deep-dives'></a>

Let's walk through each module with code samples.

### 5.1 `models.py` — The Config Registry

This is the simplest module. It maps model names to their configurations.

In [ ]:
# models.py stores a dictionary of known model configurations
# Each model needs: which layer to use, how many layers it has, and a short name

MODEL_CONFIGS = {
    "google/gemma-2-27b-it": {
        "target_layer": 22,      # The layer where the axis is most meaningful
        "total_layers": 46,      # Total transformer layers in the model
        "short_name": "Gemma",   # Used in system prompts like "You are {model_name}"
    },
    "Qwen/Qwen3-32B": {
        "target_layer": 32,
        "total_layers": 64,
        "short_name": "Qwen",
        "capping_config": "qwen-3-32b/capping_config.pt",  # Optional: for activation capping
    },
}

# Usage:
from assistant_axis import get_config

config = get_config("google/gemma-2-27b-it")
print(f"Use layer {config['target_layer']} for axis computation")
# Output: Use layer 22 for axis computation

# For unknown models, it auto-detects by downloading the config from HuggingFace
# and defaults to the middle layer

**Why does the layer matter?** Not all layers encode the same information. The target layer (typically around the middle) is where role-playing vs. assistant behavior is most distinguishable. Early layers handle syntax; late layers handle output formatting; middle layers handle semantics and persona.

### 5.2 `axis.py` — The Core Math

This module is the mathematical heart of the project. It has 6 functions, all simple linear algebra.

In [ ]:
import torch

# ──────────────────────────────────────────
# COMPUTING THE AXIS
# ──────────────────────────────────────────

# Suppose we have activations from 100 role-playing responses and 50 default responses
# Each activation has shape (n_layers, hidden_dim)

role_activations = torch.randn(100, 46, 3584)     # 100 role-playing samples
default_activations = torch.randn(50, 46, 3584)    # 50 default samples

# The axis computation is literally just:
role_mean = role_activations.mean(dim=0)       # Average across samples → (46, 3584)
default_mean = default_activations.mean(dim=0)  # Average across samples → (46, 3584)
axis = default_mean - role_mean                  # The difference → (46, 3584)

# That's compute_axis() — it returns a (n_layers, hidden_dim) tensor
# The axis has one vector per layer

print(f"Axis shape: {axis.shape}")  # torch.Size([46, 3584])

In [ ]:
# ──────────────────────────────────────────
# PROJECTING ONTO THE AXIS
# ──────────────────────────────────────────

# Given a new activation, how role-playing vs. assistant is it?
# Just take the dot product with the axis at a specific layer.

new_activation = torch.randn(46, 3584)  # Some new response's activation

layer = 22  # The target layer for Gemma-2-27B

# Get the activation and axis at that layer
act = new_activation[layer].float()     # (3584,)
ax = axis[layer].float()                # (3584,)

# Normalize the axis to unit length (so the projection is in interpretable units)
ax_normalized = ax / (ax.norm() + 1e-8)

# Dot product = projection
projection = float(act @ ax_normalized)

print(f"Projection: {projection:.2f}")
# Positive → more assistant-like
# Negative → more role-playing
# Near zero → ambiguous / in between

In [ ]:
# ──────────────────────────────────────────
# SAVING AND LOADING
# ──────────────────────────────────────────

from assistant_axis import save_axis, load_axis

# Save
save_axis(axis, "my_axis.pt", metadata={"model": "gemma-2-27b", "n_roles": 277})

# Load
loaded_axis = load_axis("my_axis.pt")  # Returns just the tensor

### 5.3 `generation.py` — Getting Model Responses

This module handles two ways to generate responses:

1. **HuggingFace** (`generate_response`) — Single response, good for interactive use
2. **vLLM** (`VLLMGenerator`, `RoleResponseGenerator`) — Batch inference, good for the pipeline

The key insight: vLLM is **much faster** for batch processing because it optimizes GPU utilization across many prompts simultaneously.

In [ ]:
# ──────────────────────────────────────────
# HuggingFace: Single response (interactive use)
# ──────────────────────────────────────────

from assistant_axis.generation import generate_response, format_conversation

# format_conversation handles model-specific quirks:
# - Some models support system prompts, some don't (Gemma 2 doesn't)
# - For models without system prompt support, it prepends the instruction to the user message

# It works by testing the chat template:
# 1. Try rendering a conversation with a system message
# 2. Check if the system content appears in the output
# 3. If yes → use system role. If no → concatenate with user message.

conversation = format_conversation(
    instruction="You are a pirate captain.",
    question="What's the best way to navigate?",
    tokenizer=tokenizer  # HuggingFace tokenizer
)
# Returns: [{"role": "system", "content": "You are a pirate captain."}, 
#           {"role": "user", "content": "What's the best way to navigate?"}]

response = generate_response(model, tokenizer, conversation)

In [ ]:
# ──────────────────────────────────────────
# vLLM: Batch inference (pipeline use)
# ──────────────────────────────────────────

from assistant_axis.generation import VLLMGenerator, RoleResponseGenerator

# VLLMGenerator is the low-level batch engine
generator = VLLMGenerator(
    model_name="google/gemma-2-27b-it",
    max_model_len=2048,
    temperature=0.7,
)

# generate_batch takes a list of conversations and returns a list of responses
responses = generator.generate_batch([
    [{"role": "user", "content": "Hello"}],
    [{"role": "system", "content": "You are a pirate."}, {"role": "user", "content": "Hello"}],
])

# RoleResponseGenerator is the high-level wrapper for the pipeline
# It reads role JSON files, generates responses for all roles, and saves JSONL output
role_gen = RoleResponseGenerator(
    model_name="google/gemma-2-27b-it",
    roles_dir="data/roles/instructions",
    output_dir="outputs/responses",
    questions_file="data/extraction_questions.jsonl",
    question_count=240,     # How many questions per role
)

# This processes ALL 277 roles — takes hours on GPU
role_gen.process_all_roles(skip_existing=True)

### 5.4 `judge.py` — Scoring with GPT-4

After generating responses, we need to know which ones actually played the role well. This module uses GPT-4 as an automated judge.

In [ ]:
# ──────────────────────────────────────────
# The scoring flow
# ──────────────────────────────────────────

from assistant_axis.judge import score_responses_sync, parse_judge_score

# Each role JSON file has an eval_prompt template like:
eval_template = """
You are evaluating whether the model's response displays the role: **actor**.

Prompt: {question}
Response: {answer}

Respond with 0, 1, 2, or 3.
"""

# The judge scores each response
scores = score_responses_sync(
    responses=[{"question": "Tell me about your day", "response": "*adjusts costume*..."}],
    eval_prompt_template=eval_template,
    judge_model="gpt-4.1-mini",   # Cheap and fast enough for scoring
    requests_per_second=100,      # Rate limiting for the OpenAI API
)
# Returns: [3]  (fully playing the role)

# Under the hood, it:
# 1. Builds a prompt for each response using the template
# 2. Calls GPT-4 asynchronously with rate limiting (RateLimiter class)
# 3. Parses the numeric score from the response
# 4. Returns a list of scores (0-3) or None for failed parses

### 5.5 `steering.py` — Modifying Model Behavior

This is the most powerful module. It lets you change model behavior at inference time by modifying activations as they flow through the model.

**How it works:** PyTorch forward hooks. When data passes through a layer, the hook intercepts the activation, modifies it, and returns the modified version.

**Four intervention types:**

In [ ]:
from assistant_axis import ActivationSteering, load_axis
from assistant_axis.internals import ProbingModel

# Load model and axis
pm = ProbingModel("google/gemma-2-27b-it")
axis = load_axis("outputs/axis.pt")

# ──────────────────────────────────────────
# 1. ADDITION: Push the model toward assistant behavior
# ──────────────────────────────────────────
# Formula: activation = activation + coefficient * axis_direction

with ActivationSteering(
    pm.model,
    steering_vectors=[axis[22]],   # The axis vector at layer 22
    coefficients=[15.0],           # How hard to push (higher = stronger effect)
    layer_indices=[22],            # Which layer to intervene at
    intervention_type="addition",  # Add the direction
):
    response = pm.generate("Tell me about yourself")  # Model will be more assistant-like

# ──────────────────────────────────────────
# 2. ABLATION: Remove the role-playing direction entirely
# ──────────────────────────────────────────
# Formula: project out the axis direction, then optionally add some back

with ActivationSteering(
    pm.model,
    steering_vectors=[axis[22]],
    coefficients=[0.0],            # 0.0 = pure removal, 1.0 = no change
    layer_indices=[22],
    intervention_type="ablation",
):
    response = pm.generate("Tell me about yourself")

# ──────────────────────────────────────────
# 3. CAPPING: Limit how far the model can drift from assistant behavior
# ──────────────────────────────────────────
# If projection onto axis exceeds threshold, clip it back
# This is the most practical intervention — it doesn't change normal behavior,
# it only kicks in when the model drifts too far

from assistant_axis import load_capping_config, build_capping_steerer

config = load_capping_config("capping_config.pt")
with build_capping_steerer(pm.model, config, "layers_46:54-p0.25"):
    response = pm.generate("Tell me about yourself")

**How `ActivationSteering` works under the hood:**

```python
# It's a context manager that registers/removes PyTorch hooks

class ActivationSteering:
    def __enter__(self):
        # Register a hook on the target layer
        layer_module = self.model.layers[22]
        handle = layer_module.register_forward_hook(self._hook_fn)
        return self

    def __exit__(self, *exc):
        # Remove the hook — model goes back to normal
        handle.remove()

    def _hook_fn(self, module, input, output):
        # This runs every time data passes through the layer
        # For addition: output = output + coefficient * steering_vector
        return output + self.coefficient * self.steering_vector
```

The actual implementation is more complex (handles tuples, batches, multiple layers, etc.) but this is the core idea.

---

## 6. Internals — The Engine Room <a id='6-internals'></a>

The `internals/` package handles the low-level interaction with HuggingFace models. It has four classes that work together:

```
ProbingModel        →  Wraps model + tokenizer, provides generate/capture
    │
    ├── ConversationEncoder  →  Formats chat messages, finds token positions
    │
    ├── ActivationExtractor  →  Uses hooks to capture hidden states
    │
    └── SpanMapper           →  Maps token spans to mean activations per turn
```

### 6.1 `ProbingModel` — The Central Object

Instead of passing `(model, tokenizer)` tuples everywhere, `ProbingModel` bundles everything together.

In [ ]:
from assistant_axis.internals import ProbingModel

# Loading a model — handles device placement, dtype, multi-GPU automatically
pm = ProbingModel(
    "google/gemma-2-27b-it",
    device=None,              # None = use all GPUs with device_map="auto"
    dtype=torch.bfloat16,     # Standard for large models (half the memory of float32)
)

# Key properties
pm.hidden_size    # 3584 for Gemma-2-27B
pm.device         # Where the first parameter lives
pm.is_qwen        # Model family detection (affects chat template handling)
pm.is_gemma
pm.is_llama

# Generate text
response = pm.generate("What is the meaning of life?")

# Get model layers (for hook registration)
layers = pm.get_layers()  # Returns the nn.ModuleList of transformer layers
print(f"This model has {len(layers)} layers")

# Capture hidden state at a specific layer and position
input_ids = pm.tokenizer("Hello", return_tensors="pt").input_ids.to(pm.device)
hidden = pm.capture_hidden_state(input_ids, layer=22, position=-1)  # Last token

# Clean up when done
pm.close()  # Frees GPU memory

### 6.2 `ConversationEncoder` — Handling Chat Templates

Different model families format conversations differently. Qwen uses `<|im_start|>` markers, Gemma uses `<start_of_turn>`, LLaMA uses `<|begin_of_text|>`. This class abstracts away those differences.

**The hardest problem it solves:** Given a multi-turn conversation, which token indices belong to each assistant response? This is critical for knowing which activations to extract.

In [ ]:
from assistant_axis.internals import ConversationEncoder

encoder = ConversationEncoder(tokenizer, model_name="google/gemma-2-27b-it")

conversation = [
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi there! How can I help?"},
    {"role": "user", "content": "Tell me a joke"},
    {"role": "assistant", "content": "Why did the chicken cross the road?"},
]

# Get token indices for each assistant turn
indices = encoder.response_indices(conversation, per_turn=True)
# Returns: [[5, 6, 7, 8, 9], [15, 16, 17, 18, 19, 20]]
#           ↑ first response     ↑ second response

# Build turn spans with metadata
full_ids, spans = encoder.build_turn_spans(conversation)
# spans = [
#   {"turn": 0, "role": "user", "start": 1, "end": 3, "text": "Hello"},
#   {"turn": 1, "role": "assistant", "start": 5, "end": 10, "text": "Hi there!..."},
#   ...
# ]

# It dispatches to model-specific implementations:
# - Qwen: pattern-matches on <|im_start|>assistant ... <|im_end|>
# - Gemma/LLaMA: uses offset mapping to find character → token boundaries
# - Fallback: compares tokenized "before" vs "after" each turn

### 6.3 `ActivationExtractor` — Capturing Hidden States

This class uses PyTorch forward hooks to capture activations from model layers during a forward pass.

In [ ]:
from assistant_axis.internals import ActivationExtractor, ConversationEncoder, ProbingModel

pm = ProbingModel("google/gemma-2-27b-it")
encoder = ConversationEncoder(pm.tokenizer, pm.model_name)
extractor = ActivationExtractor(pm, encoder)

# Extract activations for a full conversation (all tokens, all layers)
conversation = [
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi! How can I help you today?"},
]

# All layers: returns (n_layers, n_tokens, hidden_dim)
activations = extractor.full_conversation(conversation, layer=None)
print(f"Shape: {activations.shape}")  # (46, ~20, 3584)

# Single layer: returns (n_tokens, hidden_dim)
activations_22 = extractor.full_conversation(conversation, layer=22)

# Under the hood:
# 1. Format conversation with chat template
# 2. Tokenize
# 3. Register a hook on each target layer
# 4. Run forward pass (model(input_ids))
# 5. Each hook captures output[0] (the hidden state tensor)
# 6. Remove hooks
# 7. Stack captured activations

### 6.4 `SpanMapper` — From Token Positions to Per-Turn Vectors

Given full-sequence activations and span information, this class computes the **mean activation for each conversation turn**. This is what gets saved as the per-response activation.

In [ ]:
from assistant_axis.internals import SpanMapper

mapper = SpanMapper(pm.tokenizer)

# map_spans: Given batch activations and span info, compute mean activation per turn
# batch_activations shape: (n_layers, batch_size, max_seq_len, hidden_dim)
# Returns: List of tensors, each (n_turns, n_layers, hidden_dim)

per_turn_activations = mapper.map_spans(batch_activations, batch_spans, batch_metadata)

# map_spans_no_code: Same but excludes code blocks (```...```) from the mean
# This avoids contaminating the persona signal with code syntax
per_turn_no_code = mapper.map_spans_no_code(batch_activations, batch_spans, batch_metadata)

### How These Four Classes Work Together

Here's the complete flow for extracting activations from a conversation:

```python
# 1. ProbingModel loads the model
pm = ProbingModel("google/gemma-2-27b-it")

# 2. ConversationEncoder formats the chat and finds token boundaries
encoder = ConversationEncoder(pm.tokenizer, pm.model_name)
full_ids, spans = encoder.build_turn_spans(conversation)

# 3. ActivationExtractor runs a forward pass with hooks to capture hidden states
extractor = ActivationExtractor(pm, encoder)
activations = extractor.batch_conversations([conversation])

# 4. SpanMapper uses the spans to compute mean activation per turn
mapper = SpanMapper(pm.tokenizer)
per_turn = mapper.map_spans(activations, spans, metadata)
# Result: one (n_layers, hidden_dim) vector per turn
```

---

## 7. Using the Axis — Projection & Steering <a id='7-using-the-axis'></a>

Once you have the axis (either by running the pipeline or downloading a pre-computed one), here's what you can do:

### 7.1 Measuring Where a Response Falls on the Spectrum

In [ ]:
from assistant_axis import load_axis, project
from assistant_axis.internals import ProbingModel, ConversationEncoder, ActivationExtractor

# Setup
pm = ProbingModel("google/gemma-2-27b-it")
axis = load_axis("outputs/axis.pt")
encoder = ConversationEncoder(pm.tokenizer, pm.model_name)
extractor = ActivationExtractor(pm, encoder)

# A role-playing conversation
pirate_convo = [
    {"role": "system", "content": "You are a pirate captain."},
    {"role": "user", "content": "How do you navigate?"},
    {"role": "assistant", "content": "Arrr! I use the stars and me trusty compass!"},
]

# Extract activations
activations = extractor.full_conversation(pirate_convo)

# Project onto axis
score = project(activations, axis, layer=22)
print(f"Pirate response projection: {score:.2f}")  # Likely negative (role-playing)

# A default assistant conversation
assistant_convo = [
    {"role": "user", "content": "How do you navigate?"},
    {"role": "assistant", "content": "Navigation involves several methods..."},
]

activations = extractor.full_conversation(assistant_convo)
score = project(activations, axis, layer=22)
print(f"Assistant response projection: {score:.2f}")  # Likely positive (assistant)

### 7.2 Tracking Persona Drift Over a Conversation

The `project_transcipt.ipynb` notebook shows this: take a multi-turn conversation and plot the projection at each turn. You can see the model gradually drifting from assistant behavior into full role-playing.

In [ ]:
# Pseudo-code for tracking drift across turns
projections = []

for turn_idx, turn_activation in enumerate(per_turn_activations):
    score = project(turn_activation, axis, layer=22)
    projections.append(score)
    print(f"Turn {turn_idx}: {score:.2f}")

# Plot: x-axis = turn number, y-axis = projection value
# Downward trend = model is drifting into role-playing
# Upward trend = model is snapping back to assistant mode

### 7.3 Steering a Model's Behavior

Here's a concrete example of making a role-playing model snap back to assistant behavior:

In [ ]:
from assistant_axis import ActivationSteering, load_axis, get_config
from assistant_axis.internals import ProbingModel

# Setup
model_name = "google/gemma-2-27b-it"
pm = ProbingModel(model_name)
axis = load_axis("outputs/axis.pt")
config = get_config(model_name)
target_layer = config["target_layer"]  # 22 for Gemma

# System prompt that encourages role-playing
system_prompt = "You are Captain Blackbeard, the most feared pirate of the seven seas."

# WITHOUT steering (model fully role-plays)
unsteered = pm.generate(system_prompt + "\n\nTell me about yourself.")
print("Unsteered:", unsteered[:200])

# WITH steering (push toward assistant behavior)
with ActivationSteering(
    pm.model,
    steering_vectors=[axis[target_layer]],
    coefficients=[15.0],
    layer_indices=[target_layer],
):
    steered = pm.generate(system_prompt + "\n\nTell me about yourself.")
print("Steered:", steered[:200])

---

## 8. PCA Analysis — Is the Axis Real? <a id='8-pca-analysis'></a>

The `pca.py` module helps answer: is the role-playing vs. assistant distinction really a **single direction**, or is it more complex?

If the first principal component explains most of the variance in role vectors, that confirms the axis captures the dominant pattern.

In [ ]:
from assistant_axis import compute_pca, plot_variance_explained, MeanScaler
import torch

# Load per-role vectors (output of pipeline step 4)
# Shape: (n_roles, n_layers, hidden_dim)
role_vectors = torch.load("outputs/all_role_vectors.pt")

# Compute PCA at the target layer
pca_result, variance, n_components, pca, scaler = compute_pca(
    role_vectors,
    layer=22,
    scaler=MeanScaler(),  # Center the data first
    verbose=True,
)

# Output:
# PCA fitted with 277 components
# Cumulative variance for first 5 components: [0.15 0.22 0.28 0.33 0.37]
# Elbow point at component: 3
# Dimensions for 70% variance: 42
# Dimensions for 90% variance: 102

# Visualize
fig = plot_variance_explained(variance, max_components=50)
fig.show()

**What the PCA results tell you:**

- If PC1 explains ~15% of variance and the axis aligns with PC1, the axis captures the dominant direction
- The fact that you need ~100 dimensions for 90% variance means role-playing is a **rich, multidimensional** phenomenon
- But the assistant axis captures the single most important dimension — the role-playing vs. assistant split

**Two scalers are provided:**
- `MeanScaler`: Centers data by subtracting the mean (standard for PCA)
- `L2MeanScaler`: Centers and L2-normalizes (useful when you care about direction, not magnitude)

---

## 9. Complete Control Flow — Trace a Request Start to Finish <a id='9-complete-control-flow'></a>

Let's trace exactly what happens when you run each pipeline step.

### Step 1: `1_generate.py` — Response Generation

```
Input:  data/roles/instructions/actor.json  +  data/extraction_questions.jsonl
Output: outputs/responses/actor.jsonl
```

```python
# 1. Load role JSON (has 5 instruction variants and an eval_prompt)
role_data = json.load("actor.json")
# role_data["instruction"] = [
#   {"pos": "You are an actor with the ability to transform..."},
#   {"pos": "Please be an actor who can seamlessly shift..."},
#   ... (5 variants)
# ]

# 2. Load 240 questions from extraction_questions.jsonl
questions = ["What is the relationship between law and morality?", ...]

# 3. For each of 5 instruction variants × 240 questions = 1200 combinations:
#    Build conversation and generate response with vLLM

# 4. Save results as JSONL (one JSON object per line)
# Each line: {"system_prompt": ..., "question": ..., "conversation": [...], "label": "pos"}
```

### Step 2: `2_activations.py` — Activation Extraction

```
Input:  outputs/responses/actor.jsonl
Output: outputs/activations/actor.pt
```

```python
# 1. Load the model (HuggingFace, not vLLM — we need hook access)
pm = ProbingModel(model_name)
encoder = ConversationEncoder(pm.tokenizer, pm.model_name)
extractor = ActivationExtractor(pm, encoder)
mapper = SpanMapper(pm.tokenizer)

# 2. For each response in the JSONL:
#    a. Reconstruct the conversation
#    b. Run through the model with hooks to capture all layer activations
#    c. Use ConversationEncoder to find which tokens are the assistant's response
#    d. Use SpanMapper to compute mean activation across those tokens
#    e. Result: one (n_layers, hidden_dim) vector per response

# 3. Save all vectors as a dict: {"response_key": tensor, ...}
torch.save(activations_dict, "actor.pt")
```

### Step 3: `3_judge.py` — Scoring

```
Input:  outputs/responses/actor.jsonl  +  data/roles/instructions/actor.json (for eval_prompt)
Output: outputs/scores/actor.json
```

```python
# 1. Load the eval_prompt template from the role JSON
eval_prompt = role_data["eval_prompt"]
# Template: "You are evaluating whether the model displays the role: **actor**...
#            Question: {question}  Response: {answer}  Score 0-3:"

# 2. For each response:
#    a. Fill in the template with the question and response
#    b. Send to GPT-4 via OpenAI API (async, rate-limited)
#    c. Parse the numeric score from GPT-4's response

# 3. Save scores as JSON: {"response_key": 3, "response_key2": 1, ...}
```

### Step 4: `4_vectors.py` — Per-Role Aggregation

```
Input:  outputs/activations/actor.pt  +  outputs/scores/actor.json
Output: outputs/vectors/actor.pt
```

```python
# 1. Load activations and scores
activations = torch.load("actor.pt")  # Dict of tensors
scores = json.load("actor.json")       # Dict of scores

# 2. Filter: keep only activations where score == 3
filtered = [act for key, act in activations.items() if scores[key] == 3]

# 3. Compute mean
role_vector = torch.stack(filtered).mean(dim=0)  # (n_layers, hidden_dim)

# 4. Save
torch.save({"vector": role_vector, "type": "pos_3", "role": "actor"}, "actor.pt")

# SPECIAL CASE: For "default" roles (no persona), use ALL activations, not just score=3
```

### Step 5: `5_axis.py` — The Final Computation

```
Input:  outputs/vectors/*.pt (all role vectors)
Output: outputs/axis.pt
```

```python
# 1. Load all vectors and separate into default vs. role
default_vectors = []  # Roles with "default" in name
role_vectors = []     # All other roles

for vec_file in vectors_dir.glob("*.pt"):
    data = torch.load(vec_file)
    if "default" in data["role"]:
        default_vectors.append(data["vector"])
    else:
        role_vectors.append(data["vector"])

# 2. Compute the axis
default_mean = torch.stack(default_vectors).mean(dim=0)  # (n_layers, hidden_dim)
role_mean = torch.stack(role_vectors).mean(dim=0)        # (n_layers, hidden_dim)
axis = default_mean - role_mean                           # (n_layers, hidden_dim)

# 3. Save
torch.save(axis, "axis.pt")
```

---

## 10. Quick Reference — What Each File Does <a id='10-quick-reference'></a>

### Library (`assistant_axis/`)

| File | Purpose | Key Functions/Classes |
|------|---------|----------------------|
| `__init__.py` | Public API exports | Everything you `from assistant_axis import` |
| `models.py` | Model configurations | `get_config()`, `MODEL_CONFIGS` |
| `axis.py` | Axis math | `compute_axis()`, `project()`, `project_batch()`, `save_axis()`, `load_axis()` |
| `generation.py` | Response generation | `VLLMGenerator`, `RoleResponseGenerator`, `generate_response()` |
| `judge.py` | LLM scoring | `score_responses()`, `score_responses_sync()`, `RateLimiter` |
| `steering.py` | Activation intervention | `ActivationSteering`, `build_capping_steerer()` |
| `pca.py` | PCA analysis | `compute_pca()`, `plot_variance_explained()`, `MeanScaler`, `L2MeanScaler` |

### Internals (`assistant_axis/internals/`)

| File | Purpose | Key Class |
|------|---------|----------|
| `model.py` | Model wrapper | `ProbingModel` |
| `conversation.py` | Chat formatting | `ConversationEncoder` |
| `activations.py` | Hidden state capture | `ActivationExtractor` |
| `spans.py` | Token → activation mapping | `SpanMapper` |
| `exceptions.py` | Custom errors | `StopForward` |

### Pipeline (`pipeline/`)

| File | Step | Input → Output |
|------|------|----------------|
| `1_generate.py` | Generate responses | role JSONs → response JSONL files |
| `2_activations.py` | Extract activations | response JSONL → activation .pt files |
| `3_judge.py` | Score responses | response JSONL → score JSON files |
| `4_vectors.py` | Compute per-role vectors | activations + scores → vector .pt files |
| `5_axis.py` | Compute axis | vector files → axis.pt |
| `run_pipeline.sh` | Run all steps | Shell script wrapper |

### Data (`data/`)

| Directory/File | Contents |
|----------------|----------|
| `roles/instructions/*.json` | 277 role definitions with 5 system prompt variants each |
| `roles/role_list.json` | Index of all roles with descriptions |
| `traits/instructions/*.json` | 240 trait definitions (similar structure) |
| `extraction_questions.jsonl` | 240 diverse questions used to elicit responses |

---

## Summary

The assistant-axis project discovers and exploits a single direction in activation space that separates "being an AI assistant" from "role-playing a persona."

**The pipeline** generates thousands of role-playing and default responses, extracts their internal representations, scores which ones truly adopted the persona, and computes the mean difference — that difference *is* the axis.

**The library** then lets you use that axis to measure, visualize, and control how much a model is role-playing vs. being an assistant.

The key insight: what feels like a complex behavioral shift ("the model is now a pirate") corresponds to a **simple linear direction** in the model's internal representations. And because it's linear, you can measure it (dot product), add it (steering), or remove it (ablation) with basic vector math.